# P15 第二模型：CP→OR 可訓練雙塔模型

這個 notebook 將第二模型改成接近主模型流程，而不是只做簡化的 retrieval baseline。

流程如下：

1. 先以 `compatibility_train.txt` 訓練 CP binary compatibility prediction model。
2. 以 `compatibility_valid.txt` 選擇最佳 CP checkpoint。
3. 在 `compatibility_test.txt` 計算 CP AUC，並以 `fill_in_blank_test.json` 計算 CP FITB Accuracy。
4. 將 CP model 的 item encoder 與 outfit text encoder 遷移到 OR retrieval model。
5. 以 FITB ranking 訓練 OR model，並在 test split 上計算 OR/CIR Recall@K。

兩個文字條件會分開訓練、分開儲存：

- `original_text`：原始 `url_name + title`
- `context_aware_description`：本研究生成之情境感知描述

輸出根目錄維持不變：

```text
/home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/
```

In [1]:
from pathlib import Path
import json
import pickle
import csv
import math
import random
import time
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else (lambda x: x)

try:
    from sklearn.metrics import roc_auc_score as sklearn_roc_auc_score
except Exception:
    sklearn_roc_auc_score = None

# ------------------------------------------------------------
# 路徑設定
# ------------------------------------------------------------
PROJECT_ROOT = Path('/home/ester/valid_description')
POLYVORE_DIR = PROJECT_ROOT / 'polyvore_data' / 'polyvore_outfits'
DISJOINT_DIR = POLYVORE_DIR / 'disjoint'
FASHIONCLIP_DIR = PROJECT_ROOT / 'fashionclip_data'
JOURNAL_DIR = PROJECT_ROOT / 'experiment data' / 'journal_figures_tables_20260629'

RUN_ROOT = JOURNAL_DIR / 'second_model_runs' / 'fashionclip_two_tower_fitb_or'
MODEL_ROOT = RUN_ROOT / 'models'
RESULT_ROOT = RUN_ROOT / 'results'

for directory in [RUN_ROOT, MODEL_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

print('第二模型輸出根目錄：')
print(RUN_ROOT)

第二模型輸出根目錄：
/home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or


In [2]:
# ------------------------------------------------------------
# 實驗設定
# ------------------------------------------------------------

# 新流程建議先跑 seed 1，確認 CP AUC / FITB / OR Recall 都正常後，再改回 [1, 2, 3, 4, 5]。
SEEDS = [1, 2, 3, 4, 5]

TEXT_VARIANTS = {
    'original_text': 'encoded_outfitUrlTitle_en_fashionClip.pkl',
    'context_aware_description': 'encoded_NewoutfitUrlTitle_en_fashionClip.pkl',
}

MAX_ITEM_LEN = 16
ITEM_DIM = 1024       # FashionCLIP image 512 + item text 512
TEXT_DIM = 512        # outfit-level FashionCLIP text embedding
CATEGORY_DIM = 512    # category text embedding

# 與 Hybrid Attention 主模型對齊：最多訓練 100 epochs，不使用 early stopping。
# CP 與 OR 都以 validation FITB accuracy 更新最佳 checkpoint。
CP_EPOCHS = 100
OR_EPOCHS = 100

BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.15
HIDDEN_DIM = 512
EMBED_DIM = 256
NUM_WORKERS = 0

MIN_CANDIDATES_PER_FG = 3000
K_LIST = [1, 3, 5, 10, 30, 50]

# True 會覆蓋同一路徑下舊的 checkpoint / log / result。
FORCE_RETRAIN = True

# 除錯用：正式實驗請維持 None。
TRAIN_LIMIT = None
VALID_LIMIT = None
TEST_LIMIT = None

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PIN_MEMORY = DEVICE == 'cuda'

print('使用裝置：', DEVICE)
print('訓練 seeds：', SEEDS)
print('文字條件：', list(TEXT_VARIANTS.keys()))

使用裝置： cuda
訓練 seeds： [1, 2, 3, 4, 5]
文字條件： ['original_text', 'context_aware_description']


In [3]:
# ------------------------------------------------------------
# 基本工具
# ------------------------------------------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_pickle(path: Path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def as_key(x) -> str:
    return str(x)


def vector_to_float32(x) -> np.ndarray:
    return np.asarray(x, dtype=np.float32)


def write_csv(path: Path, rows: List[dict], fieldnames: Optional[List[str]] = None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = sorted({key for row in rows for key in row.keys()}) if rows else []
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    print(f'[寫入] {path}')


def append_csv(path: Path, row: dict, fieldnames: List[str]):
    path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = path.exists()
    with open(path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def mean_std_text(values: List[float], digits: int = 4) -> str:
    arr = np.asarray(values, dtype=np.float64)
    if len(arr) == 0:
        return ''
    if len(arr) == 1:
        return f'{arr.mean():.{digits}f} ± 0.0000'
    return f'{arr.mean():.{digits}f} ± {arr.std(ddof=1):.{digits}f}'


def binary_auc(labels, scores) -> float:
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if sklearn_roc_auc_score is not None:
        return float(sklearn_roc_auc_score(labels, scores))

    order = np.argsort(scores)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(scores) + 1)
    pos = labels == 1
    n_pos = pos.sum()
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float('nan')
    rank_sum_pos = ranks[pos].sum()
    auc = (rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return float(auc)

In [4]:
# ------------------------------------------------------------
# 載入資料與特徵
# ------------------------------------------------------------
required_paths = [
    FASHIONCLIP_DIR / 'img_feats_fashionClip.pkl',
    FASHIONCLIP_DIR / 'encoded_title_description_distiluse-base-multilingual-cased-v2.pkl',
    FASHIONCLIP_DIR / 'encoded_category_distiluse-base-multilingual-cased-v2.pkl',
    FASHIONCLIP_DIR / TEXT_VARIANTS['original_text'],
    FASHIONCLIP_DIR / TEXT_VARIANTS['context_aware_description'],
    POLYVORE_DIR / 'polyvore_item_metadata.json',
    DISJOINT_DIR / 'train.json',
    DISJOINT_DIR / 'valid.json',
    DISJOINT_DIR / 'test.json',
    DISJOINT_DIR / 'compatibility_train.txt',
    DISJOINT_DIR / 'compatibility_valid.txt',
    DISJOINT_DIR / 'compatibility_test.txt',
    DISJOINT_DIR / 'fill_in_blank_train.json',
    DISJOINT_DIR / 'fill_in_blank_valid.json',
    DISJOINT_DIR / 'fill_in_blank_test.json',
]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError('找不到必要檔案：\n' + '\n'.join(missing_paths))

img_feats = load_pickle(FASHIONCLIP_DIR / 'img_feats_fashionClip.pkl')
item_text_feats = load_pickle(FASHIONCLIP_DIR / 'encoded_title_description_distiluse-base-multilingual-cased-v2.pkl')
category_feats = load_pickle(FASHIONCLIP_DIR / 'encoded_category_distiluse-base-multilingual-cased-v2.pkl')
metadata = load_json(POLYVORE_DIR / 'polyvore_item_metadata.json')

split_outfits = {
    'train': load_json(DISJOINT_DIR / 'train.json'),
    'valid': load_json(DISJOINT_DIR / 'valid.json'),
    'test': load_json(DISJOINT_DIR / 'test.json'),
}
fitb_raw = {
    'train': load_json(DISJOINT_DIR / 'fill_in_blank_train.json'),
    'valid': load_json(DISJOINT_DIR / 'fill_in_blank_valid.json'),
    'test': load_json(DISJOINT_DIR / 'fill_in_blank_test.json'),
}

print('PO-D outfit 數量：')
for split in ['train', 'valid', 'test']:
    print(f'  {split}: outfits={len(split_outfits[split]):,}, FITB questions={len(fitb_raw[split]):,}')
print(f'item image features: {len(img_feats):,}')
print(f'item text features: {len(item_text_feats):,}')
print(f'category features: {len(category_feats):,}')

PO-D outfit 數量：
  train: outfits=16,995, FITB questions=16,995
  valid: outfits=3,000, FITB questions=3,000
  test: outfits=15,145, FITB questions=15,145
item image features: 251,008
item text features: 251,008
category features: 251,008


In [5]:
# ------------------------------------------------------------
# 題目解析
# ------------------------------------------------------------
def build_ref_to_item(outfits: List[dict]) -> Dict[str, str]:
    ref_to_item = {}
    for outfit in outfits:
        set_id = as_key(outfit['set_id'])
        for item in outfit['items']:
            ref_to_item[f'{set_id}_{item["index"]}'] = as_key(item['item_id'])
    return ref_to_item


ref_to_item_by_split = {split: build_ref_to_item(outfits) for split, outfits in split_outfits.items()}


def parse_compatibility_examples(split: str, limit: Optional[int] = None) -> List[dict]:
    path = DISJOINT_DIR / f'compatibility_{split}.txt'
    ref_to_item = ref_to_item_by_split[split]
    examples = []
    skipped = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line_idx, line in enumerate(f):
            parts = line.strip().split()
            if len(parts) < 2:
                skipped += 1
                continue
            label = int(parts[0])
            refs = parts[1:]
            set_id = as_key(refs[0].split('_')[0])
            try:
                item_ids = [ref_to_item[ref] for ref in refs]
            except KeyError:
                skipped += 1
                continue
            examples.append({
                'question_index': line_idx,
                'set_id': set_id,
                'item_ids': item_ids,
                'label': label,
            })
            if limit is not None and len(examples) >= limit:
                break
    print(f'compatibility {split}: parsed={len(examples):,}, skipped={skipped:,}')
    return examples


def parse_fitb_examples(split: str, limit: Optional[int] = None) -> List[dict]:
    ref_to_item = ref_to_item_by_split[split]
    examples = []
    skipped = 0
    for q_idx, row in enumerate(fitb_raw[split]):
        question_refs = row['question']
        answer_refs = row['answers']
        if not question_refs or not answer_refs:
            skipped += 1
            continue
        set_id = as_key(question_refs[0].split('_')[0])
        try:
            partial_ids = [ref_to_item[ref] for ref in question_refs]
            answer_ids = [ref_to_item[ref] for ref in answer_refs]
        except KeyError:
            skipped += 1
            continue
        correct_indices = [i for i, ref in enumerate(answer_refs) if as_key(ref.split('_')[0]) == set_id]
        if len(correct_indices) != 1:
            skipped += 1
            continue
        correct_idx = correct_indices[0]
        target_item_id = answer_ids[correct_idx]
        if target_item_id not in metadata:
            skipped += 1
            continue
        examples.append({
            'question_index': q_idx,
            'set_id': set_id,
            'blank_position': int(row.get('blank_position', -1)),
            'partial_item_ids': partial_ids,
            'answer_item_ids': answer_ids,
            'correct_index': correct_idx,
            'target_item_id': target_item_id,
            'target_fg': as_key(metadata[target_item_id]['category_id']),
            'target_semantic_category': as_key(metadata[target_item_id].get('semantic_category', '')),
        })
        if limit is not None and len(examples) >= limit:
            break
    print(f'FITB {split}: parsed={len(examples):,}, skipped={skipped:,}')
    return examples


compat_examples_by_split = {
    'train': parse_compatibility_examples('train', TRAIN_LIMIT),
    'valid': parse_compatibility_examples('valid', VALID_LIMIT),
    'test': parse_compatibility_examples('test', TEST_LIMIT),
}
fitb_examples_by_split = {
    'train': parse_fitb_examples('train', TRAIN_LIMIT),
    'valid': parse_fitb_examples('valid', VALID_LIMIT),
    'test': parse_fitb_examples('test', TEST_LIMIT),
}

compatibility train: parsed=33,990, skipped=0
compatibility valid: parsed=6,000, skipped=0
compatibility test: parsed=30,290, skipped=0
FITB train: parsed=16,995, skipped=0
FITB valid: parsed=3,000, skipped=0
FITB test: parsed=15,145, skipped=0


In [6]:
# ------------------------------------------------------------
# Feature 組合與 Dataset
# ------------------------------------------------------------
_item_vec_cache: Dict[str, np.ndarray] = {}


def get_item_vec(item_id: str) -> np.ndarray:
    item_id = as_key(item_id)
    if item_id not in _item_vec_cache:
        img = vector_to_float32(img_feats[item_id])
        txt = vector_to_float32(item_text_feats[item_id])
        _item_vec_cache[item_id] = np.concatenate([img, txt]).astype(np.float32)
    return _item_vec_cache[item_id]


def get_category_vec(item_id: str) -> np.ndarray:
    return vector_to_float32(category_feats[as_key(item_id)])


def pad_items(item_ids: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    arr = np.zeros((MAX_ITEM_LEN, ITEM_DIM), dtype=np.float32)
    mask = np.zeros((MAX_ITEM_LEN,), dtype=np.float32)
    keep = item_ids[:MAX_ITEM_LEN]
    for i, item_id in enumerate(keep):
        arr[i] = get_item_vec(item_id)
        mask[i] = 1.0
    return arr, mask


class CPCompatibilityDataset(Dataset):
    def __init__(self, examples: List[dict], outfit_feats: Dict[str, np.ndarray]):
        self.examples = [ex for ex in examples if ex['set_id'] in outfit_feats]
        self.outfit_feats = outfit_feats

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index: int):
        ex = self.examples[index]
        items, mask = pad_items(ex['item_ids'])
        outfit_text = vector_to_float32(self.outfit_feats[ex['set_id']])
        return {
            'items': torch.from_numpy(items),
            'mask': torch.from_numpy(mask),
            'outfit_text': torch.from_numpy(outfit_text),
            'label': torch.tensor(float(ex['label']), dtype=torch.float32),
        }


class ORFITBDataset(Dataset):
    def __init__(self, examples: List[dict], outfit_feats: Dict[str, np.ndarray]):
        self.examples = [ex for ex in examples if ex['set_id'] in outfit_feats]
        self.outfit_feats = outfit_feats

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index: int):
        ex = self.examples[index]
        partial_items, partial_mask = pad_items(ex['partial_item_ids'])
        candidates = np.stack([get_item_vec(item_id) for item_id in ex['answer_item_ids']], axis=0).astype(np.float32)
        outfit_text = vector_to_float32(self.outfit_feats[ex['set_id']])
        category = get_category_vec(ex['target_item_id'])
        return {
            'partial_items': torch.from_numpy(partial_items),
            'partial_mask': torch.from_numpy(partial_mask),
            'outfit_text': torch.from_numpy(outfit_text),
            'category': torch.from_numpy(category),
            'candidates': torch.from_numpy(candidates),
            'label': torch.tensor(int(ex['correct_index']), dtype=torch.long),
        }


def describe_variant_coverage(variant_key: str, outfit_feats: Dict[str, np.ndarray]):
    print(f'\n[{variant_key}] outfit text embedding 筆數：{len(outfit_feats):,}')
    for split, examples in compat_examples_by_split.items():
        missing = sum(1 for ex in examples if ex['set_id'] not in outfit_feats)
        print(f'  CP {split}: usable={len(examples)-missing:,}, missing={missing:,}')
    for split, examples in fitb_examples_by_split.items():
        missing = sum(1 for ex in examples if ex['set_id'] not in outfit_feats)
        print(f'  FITB {split}: usable={len(examples)-missing:,}, missing={missing:,}')

In [7]:
# ------------------------------------------------------------
# 候選池建立：OR/CIR Recall@K
# ------------------------------------------------------------
def build_fg_to_unique_items(outfits: List[dict]) -> Dict[str, List[str]]:
    fg_to_items = defaultdict(list)
    seen_by_fg = defaultdict(set)
    for outfit in outfits:
        for item in outfit['items']:
            item_id = as_key(item['item_id'])
            if item_id not in metadata:
                continue
            fg = as_key(metadata[item_id]['category_id'])
            if item_id not in seen_by_fg[fg]:
                seen_by_fg[fg].add(item_id)
                fg_to_items[fg].append(item_id)
    return dict(fg_to_items)


def build_or_candidate_pools(min_candidates: int = 3000) -> Dict[str, List[str]]:
    test_fg = build_fg_to_unique_items(split_outfits['test'])
    train_fg = build_fg_to_unique_items(split_outfits['train'])
    pools = {}
    for fg, test_items in test_fg.items():
        seen = set()
        pool = []
        for item_id in test_items:
            if item_id not in seen:
                seen.add(item_id)
                pool.append(item_id)
                if len(pool) >= min_candidates:
                    break
        if len(pool) < min_candidates:
            for item_id in train_fg.get(fg, []):
                if item_id not in seen:
                    seen.add(item_id)
                    pool.append(item_id)
                    if len(pool) >= min_candidates:
                        break
        if len(pool) >= min_candidates:
            pools[fg] = pool[:min_candidates]
    print(f'OR/CIR 可評估子類別數：{len(pools):,} / test 子類別數：{len(test_fg):,}')
    return pools


candidate_pools = build_or_candidate_pools(MIN_CANDIDATES_PER_FG)
evaluable_test = [ex for ex in fitb_examples_by_split['test'] if ex['target_fg'] in candidate_pools]
print(f'測試集 FITB 題目：{len(fitb_examples_by_split["test"]):,}')
print(f'OR/CIR Recall 可評估 query-target pairs：{len(evaluable_test):,}')

OR/CIR 可評估子類別數：19 / test 子類別數：152
測試集 FITB 題目：15,145
OR/CIR Recall 可評估 query-target pairs：9,311


In [8]:
# ------------------------------------------------------------
# 模型定義
# ------------------------------------------------------------
class MLPEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, embed_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def masked_mean(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    mask = mask.unsqueeze(-1)
    denom = mask.sum(dim=1).clamp_min(1.0)
    return (x * mask).sum(dim=1) / denom


class CPCompatibilityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.item_encoder = MLPEncoder(ITEM_DIM, HIDDEN_DIM, EMBED_DIM, DROPOUT)
        self.outfit_text_encoder = MLPEncoder(TEXT_DIM, HIDDEN_DIM, EMBED_DIM, DROPOUT)
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM * 2, HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, 1),
        )

    def encode_outfit(self, items: torch.Tensor, mask: torch.Tensor, outfit_text: torch.Tensor) -> torch.Tensor:
        item_z = self.item_encoder(items)
        outfit_z = masked_mean(item_z, mask)
        text_z = self.outfit_text_encoder(outfit_text)
        return torch.cat([outfit_z, text_z], dim=-1)

    def forward(self, items: torch.Tensor, mask: torch.Tensor, outfit_text: torch.Tensor) -> torch.Tensor:
        z = self.encode_outfit(items, mask, outfit_text)
        return self.classifier(z).squeeze(-1)


class ORRetrievalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.item_encoder = MLPEncoder(ITEM_DIM, HIDDEN_DIM, EMBED_DIM, DROPOUT)
        self.outfit_text_encoder = MLPEncoder(TEXT_DIM, HIDDEN_DIM, EMBED_DIM, DROPOUT)
        self.category_encoder = MLPEncoder(CATEGORY_DIM, HIDDEN_DIM, EMBED_DIM, DROPOUT)
        self.query_net = nn.Sequential(
            nn.Linear(EMBED_DIM * 3, HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, EMBED_DIM),
        )
        self.logit_scale = nn.Parameter(torch.tensor(math.log(10.0), dtype=torch.float32))

    def encode_query(self, partial_items: torch.Tensor, partial_mask: torch.Tensor, outfit_text: torch.Tensor, category: torch.Tensor) -> torch.Tensor:
        partial_item_z = self.item_encoder(partial_items)
        partial_z = masked_mean(partial_item_z, partial_mask)
        text_z = self.outfit_text_encoder(outfit_text)
        category_z = self.category_encoder(category)
        query = self.query_net(torch.cat([partial_z, text_z, category_z], dim=-1))
        return F.normalize(query, dim=-1)

    def encode_candidate(self, candidates: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.item_encoder(candidates), dim=-1)

    def forward(self, partial_items: torch.Tensor, partial_mask: torch.Tensor, outfit_text: torch.Tensor, category: torch.Tensor, candidates: torch.Tensor) -> torch.Tensor:
        query = self.encode_query(partial_items, partial_mask, outfit_text, category)
        cand = self.encode_candidate(candidates)
        scale = self.logit_scale.exp().clamp(max=100.0)
        return torch.einsum('bd,bkd->bk', query, cand) * scale


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool, seed: int) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=generator if shuffle else None,
    )

In [9]:
# ------------------------------------------------------------
# CP 訓練與評估
# ------------------------------------------------------------
def evaluate_cp_auc(model: CPCompatibilityModel, loader: DataLoader) -> float:
    model.eval()
    labels = []
    scores = []
    with torch.no_grad():
        for batch in loader:
            items = batch['items'].to(DEVICE).float()
            mask = batch['mask'].to(DEVICE).float()
            outfit_text = batch['outfit_text'].to(DEVICE).float()
            label = batch['label'].detach().cpu().numpy()
            logit = model(items, mask, outfit_text).detach().cpu().numpy()
            labels.extend(label.tolist())
            scores.extend(logit.tolist())
    return binary_auc(labels, scores)


def evaluate_cp_fitb(model: CPCompatibilityModel, examples: List[dict], outfit_feats: Dict[str, np.ndarray]) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for ex in tqdm(examples, desc='CP FITB'):
            if ex['set_id'] not in outfit_feats:
                continue
            batch_items = []
            batch_mask = []
            for answer_id in ex['answer_item_ids']:
                items, mask = pad_items(ex['partial_item_ids'] + [answer_id])
                batch_items.append(items)
                batch_mask.append(mask)
            items_t = torch.from_numpy(np.stack(batch_items)).to(DEVICE).float()
            mask_t = torch.from_numpy(np.stack(batch_mask)).to(DEVICE).float()
            outfit_text = vector_to_float32(outfit_feats[ex['set_id']])
            outfit_text_t = torch.from_numpy(np.stack([outfit_text] * len(batch_items))).to(DEVICE).float()
            scores = model(items_t, mask_t, outfit_text_t)
            pred = int(torch.argmax(scores).item())
            correct += int(pred == ex['correct_index'])
            total += 1
    return correct / total if total else 0.0


def train_cp_one_seed(variant_key: str, outfit_feats: Dict[str, np.ndarray], seed: int) -> Tuple[Path, CPCompatibilityModel, dict]:
    set_seed(seed)
    model_dir = MODEL_ROOT / variant_key / f'seed_{seed}'
    model_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = model_dir / 'cp_best_model.pt'
    log_path = model_dir / 'cp_train_log.csv'

    if ckpt_path.exists() and not FORCE_RETRAIN:
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model = CPCompatibilityModel().to(DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        return ckpt_path, model, checkpoint

    if log_path.exists():
        log_path.unlink()

    train_dataset = CPCompatibilityDataset(compat_examples_by_split['train'], outfit_feats)
    valid_dataset = CPCompatibilityDataset(compat_examples_by_split['valid'], outfit_feats)
    train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True, seed=seed)
    valid_loader = make_loader(valid_dataset, BATCH_SIZE, shuffle=False, seed=seed)

    model = CPCompatibilityModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_fitb = -1.0
    best_epoch = -1
    log_fields = ['variant', 'seed', 'stage', 'epoch', 'train_loss', 'valid_auc', 'valid_fitb_acc', 'lr', 'seconds']
    start_all = time.time()

    print(f'\n[CP 訓練] variant={variant_key}, seed={seed}, train={len(train_dataset):,}, valid={len(valid_dataset):,}')
    for epoch in range(1, CP_EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        total = 0
        epoch_start = time.time()
        for batch in tqdm(train_loader, desc=f'CP train {variant_key} seed={seed} epoch={epoch}'):
            items = batch['items'].to(DEVICE).float()
            mask = batch['mask'].to(DEVICE).float()
            outfit_text = batch['outfit_text'].to(DEVICE).float()
            label = batch['label'].to(DEVICE).float()

            optimizer.zero_grad(set_to_none=True)
            logit = model(items, mask, outfit_text)
            loss = F.binary_cross_entropy_with_logits(logit, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            batch_n = int(label.numel())
            loss_sum += float(loss.item()) * batch_n
            total += batch_n

        train_loss = loss_sum / total if total else 0.0
        valid_auc = evaluate_cp_auc(model, valid_loader)
        valid_fitb = evaluate_cp_fitb(model, fitb_examples_by_split['valid'], outfit_feats)
        scheduler.step(valid_fitb)
        lr_now = optimizer.param_groups[0]['lr']
        seconds = time.time() - epoch_start

        append_csv(log_path, {
            'variant': variant_key, 'seed': seed, 'stage': 'CP', 'epoch': epoch,
            'train_loss': f'{train_loss:.6f}', 'valid_auc': f'{valid_auc:.6f}',
            'valid_fitb_acc': f'{valid_fitb:.6f}',
            'lr': f'{lr_now:.8f}', 'seconds': f'{seconds:.2f}',
        }, log_fields)
        print(f'CP epoch={epoch:02d} loss={train_loss:.4f} valid_AUC={valid_auc:.4f} valid_FITB={valid_fitb:.4f} lr={lr_now:.2e}')

        if valid_fitb > best_fitb:
            best_fitb = valid_fitb
            best_epoch = epoch
            torch.save({
                'variant': variant_key,
                'seed': seed,
                'stage': 'CP',
                'epoch': epoch,
                'best_valid_fitb_acc': best_fitb,
                'valid_auc_at_best_fitb': valid_auc,
                'model_state_dict': model.state_dict(),
                'selection_rule': 'best checkpoint selected by validation FITB accuracy',
            }, ckpt_path)

    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    checkpoint['total_training_seconds'] = time.time() - start_all
    checkpoint['best_epoch'] = best_epoch
    return ckpt_path, model, checkpoint


In [10]:
# ------------------------------------------------------------
# OR 訓練與評估
# ------------------------------------------------------------
def evaluate_or_fitb(model: ORRetrievalModel, loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            partial_items = batch['partial_items'].to(DEVICE).float()
            partial_mask = batch['partial_mask'].to(DEVICE).float()
            outfit_text = batch['outfit_text'].to(DEVICE).float()
            category = batch['category'].to(DEVICE).float()
            candidates = batch['candidates'].to(DEVICE).float()
            label = batch['label'].to(DEVICE)
            scores = model(partial_items, partial_mask, outfit_text, category, candidates)
            pred = torch.argmax(scores, dim=1)
            correct += int((pred == label).sum().item())
            total += int(label.numel())
    return correct / total if total else 0.0


def train_or_one_seed(variant_key: str, outfit_feats: Dict[str, np.ndarray], seed: int, cp_model: CPCompatibilityModel) -> Tuple[Path, ORRetrievalModel, dict]:
    set_seed(seed)
    model_dir = MODEL_ROOT / variant_key / f'seed_{seed}'
    model_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = model_dir / 'best_model.pt'
    log_path = model_dir / 'or_train_log.csv'

    if ckpt_path.exists() and not FORCE_RETRAIN:
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model = ORRetrievalModel().to(DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        return ckpt_path, model, checkpoint

    if log_path.exists():
        log_path.unlink()

    train_dataset = ORFITBDataset(fitb_examples_by_split['train'], outfit_feats)
    valid_dataset = ORFITBDataset(fitb_examples_by_split['valid'], outfit_feats)
    train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True, seed=seed)
    valid_loader = make_loader(valid_dataset, BATCH_SIZE, shuffle=False, seed=seed)

    model = ORRetrievalModel().to(DEVICE)
    model.item_encoder.load_state_dict(cp_model.item_encoder.state_dict())
    model.outfit_text_encoder.load_state_dict(cp_model.outfit_text_encoder.state_dict())

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_fitb = -1.0
    best_epoch = -1
    log_fields = ['variant', 'seed', 'stage', 'epoch', 'train_loss', 'valid_fitb_acc', 'lr', 'seconds']
    start_all = time.time()

    print(f'\n[OR 訓練] variant={variant_key}, seed={seed}, train={len(train_dataset):,}, valid={len(valid_dataset):,}')
    for epoch in range(1, OR_EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        total = 0
        epoch_start = time.time()
        for batch in tqdm(train_loader, desc=f'OR train {variant_key} seed={seed} epoch={epoch}'):
            partial_items = batch['partial_items'].to(DEVICE).float()
            partial_mask = batch['partial_mask'].to(DEVICE).float()
            outfit_text = batch['outfit_text'].to(DEVICE).float()
            category = batch['category'].to(DEVICE).float()
            candidates = batch['candidates'].to(DEVICE).float()
            label = batch['label'].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            scores = model(partial_items, partial_mask, outfit_text, category, candidates)
            loss = F.cross_entropy(scores, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            batch_n = int(label.numel())
            loss_sum += float(loss.item()) * batch_n
            total += batch_n

        train_loss = loss_sum / total if total else 0.0
        valid_fitb = evaluate_or_fitb(model, valid_loader)
        scheduler.step(valid_fitb)
        lr_now = optimizer.param_groups[0]['lr']
        seconds = time.time() - epoch_start

        append_csv(log_path, {
            'variant': variant_key, 'seed': seed, 'stage': 'OR', 'epoch': epoch,
            'train_loss': f'{train_loss:.6f}', 'valid_fitb_acc': f'{valid_fitb:.6f}',
            'lr': f'{lr_now:.8f}', 'seconds': f'{seconds:.2f}',
        }, log_fields)
        print(f'OR epoch={epoch:02d} loss={train_loss:.4f} valid_FITB={valid_fitb:.4f} lr={lr_now:.2e}')

        if valid_fitb > best_fitb:
            best_fitb = valid_fitb
            best_epoch = epoch
            torch.save({
                'variant': variant_key,
                'seed': seed,
                'stage': 'OR',
                'epoch': epoch,
                'best_valid_fitb_acc': best_fitb,
                'model_state_dict': model.state_dict(),
                'selection_rule': 'best checkpoint selected by validation FITB accuracy',
                'transfer_note': 'item_encoder and outfit_text_encoder initialized from CP model',
            }, ckpt_path)

    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    checkpoint['total_training_seconds'] = time.time() - start_all
    checkpoint['best_epoch'] = best_epoch
    return ckpt_path, model, checkpoint


In [11]:
# ------------------------------------------------------------
# OR/CIR Recall@K 測試
# ------------------------------------------------------------
def encode_candidates(model: ORRetrievalModel, item_ids: List[str], batch_size: int = 1024) -> torch.Tensor:
    encoded_batches = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(item_ids), batch_size):
            batch_ids = item_ids[start:start + batch_size]
            raw = np.stack([get_item_vec(item_id) for item_id in batch_ids], axis=0).astype(np.float32)
            raw_t = torch.from_numpy(raw).to(DEVICE).float()
            encoded_batches.append(model.encode_candidate(raw_t))
    return torch.cat(encoded_batches, dim=0)


def encode_one_query(model: ORRetrievalModel, ex: dict, outfit_feats: Dict[str, np.ndarray]) -> torch.Tensor:
    partial_items, partial_mask = pad_items(ex['partial_item_ids'])
    outfit_text = vector_to_float32(outfit_feats[ex['set_id']])
    category = get_category_vec(ex['target_item_id'])
    with torch.no_grad():
        return model.encode_query(
            torch.from_numpy(partial_items).unsqueeze(0).to(DEVICE).float(),
            torch.from_numpy(partial_mask).unsqueeze(0).to(DEVICE).float(),
            torch.from_numpy(outfit_text).unsqueeze(0).to(DEVICE).float(),
            torch.from_numpy(category).unsqueeze(0).to(DEVICE).float(),
        ).squeeze(0)


def evaluate_or_recall(model: ORRetrievalModel, outfit_feats: Dict[str, np.ndarray], variant_key: str, seed: int) -> Tuple[dict, List[dict]]:
    model.eval()
    pool_emb_cache = {}
    pool_id_cache = {}
    for fg, ids in tqdm(candidate_pools.items(), desc=f'encode candidate pools [{variant_key} seed={seed}]'):
        pool_id_cache[fg] = list(ids)
        pool_emb_cache[fg] = encode_candidates(model, ids)

    target_emb_cache: Dict[str, torch.Tensor] = {}
    rows = []
    hits = {k: 0 for k in K_LIST}
    ranks = []
    skipped = 0

    for ex in tqdm(fitb_examples_by_split['test'], desc=f'OR/CIR recall [{variant_key} seed={seed}]'):
        if ex['set_id'] not in outfit_feats:
            skipped += 1
            continue
        fg = ex['target_fg']
        target_item_id = ex['target_item_id']
        if fg not in candidate_pools:
            skipped += 1
            continue

        ids = list(pool_id_cache[fg])
        cand_z = pool_emb_cache[fg]
        if target_item_id in ids:
            target_pos = ids.index(target_item_id)
        else:
            ids = ids[:-1] + [target_item_id]
            if target_item_id not in target_emb_cache:
                target_emb_cache[target_item_id] = encode_candidates(model, [target_item_id])
            cand_z = torch.cat([cand_z[:-1], target_emb_cache[target_item_id]], dim=0)
            target_pos = len(ids) - 1

        q_z = encode_one_query(model, ex, outfit_feats)
        scores = torch.matmul(cand_z, q_z)
        target_score = scores[target_pos]
        rank = int((scores > target_score).sum().item()) + 1
        ranks.append(rank)

        top_k = min(max(K_LIST), len(ids))
        _, top_idx = torch.topk(scores, k=top_k, largest=True)
        top_ids = [ids[int(i)] for i in top_idx.detach().cpu().tolist()]

        row = {
            'variant': variant_key,
            'seed': seed,
            'question_index': ex['question_index'],
            'set_id': ex['set_id'],
            'target_item_id': target_item_id,
            'target_fg': fg,
            'target_semantic_category': ex['target_semantic_category'],
            'rank': rank,
            'partial_item_ids': json.dumps(ex['partial_item_ids'], ensure_ascii=False),
            'answer_item_ids': json.dumps(ex['answer_item_ids'], ensure_ascii=False),
            'top10_ids': json.dumps(top_ids[:10], ensure_ascii=False),
        }
        for k in K_LIST:
            hit = int(rank <= k)
            hits[k] += hit
            row[f'recall@{k}'] = hit
        rows.append(row)

    n = len(rows)
    summary = {
        'variant': variant_key,
        'seed': seed,
        'test_fitb_questions': len(fitb_examples_by_split['test']),
        'or_recall_evaluable_pairs': n,
        'or_recall_skipped_questions': skipped,
        'mean_rank': float(np.mean(ranks)) if ranks else float('nan'),
        'median_rank': float(np.median(ranks)) if ranks else float('nan'),
    }
    for k in K_LIST:
        summary[f'recall@{k}'] = hits[k] / n if n else 0.0
    return summary, rows

In [12]:
# ------------------------------------------------------------
# 執行完整 CP→OR 流程
# ------------------------------------------------------------
def clean_seed_outputs(variant_key: str, seed: int):
    model_dir = MODEL_ROOT / variant_key / f'seed_{seed}'
    result_dir = RESULT_ROOT / variant_key / f'seed_{seed}'
    if FORCE_RETRAIN:
        for path in [
            model_dir / 'train_log.csv',
            model_dir / 'cp_train_log.csv',
            model_dir / 'or_train_log.csv',
            model_dir / 'config.json',
            model_dir / 'best_model.pt',
            model_dir / 'cp_best_model.pt',
            result_dir / 'seed_summary.csv',
            result_dir / 'or_recall_rowlevel.csv',
        ]:
            if path.exists():
                path.unlink()


all_seed_rows = []
all_or_rows = []

for variant_key, feature_file in TEXT_VARIANTS.items():
    outfit_feats = load_pickle(FASHIONCLIP_DIR / feature_file)
    describe_variant_coverage(variant_key, outfit_feats)

    for seed in SEEDS:
        clean_seed_outputs(variant_key, seed)
        model_dir = MODEL_ROOT / variant_key / f'seed_{seed}'
        result_dir = RESULT_ROOT / variant_key / f'seed_{seed}'
        model_dir.mkdir(parents=True, exist_ok=True)
        result_dir.mkdir(parents=True, exist_ok=True)

        cp_ckpt_path, cp_model, cp_checkpoint = train_cp_one_seed(variant_key, outfit_feats, seed)

        cp_test_dataset = CPCompatibilityDataset(compat_examples_by_split['test'], outfit_feats)
        cp_test_loader = make_loader(cp_test_dataset, BATCH_SIZE, shuffle=False, seed=seed)
        cp_test_auc = evaluate_cp_auc(cp_model, cp_test_loader)
        cp_test_fitb = evaluate_cp_fitb(cp_model, fitb_examples_by_split['test'], outfit_feats)

        or_ckpt_path, or_model, or_checkpoint = train_or_one_seed(variant_key, outfit_feats, seed, cp_model)

        or_test_dataset = ORFITBDataset(fitb_examples_by_split['test'], outfit_feats)
        or_test_loader = make_loader(or_test_dataset, BATCH_SIZE, shuffle=False, seed=seed)
        or_test_fitb = evaluate_or_fitb(or_model, or_test_loader)

        or_summary, or_rows = evaluate_or_recall(or_model, outfit_feats, variant_key, seed)
        seed_summary = {
            'variant': variant_key,
            'seed': seed,
            'cp_best_epoch': int(cp_checkpoint.get('epoch', cp_checkpoint.get('best_epoch', -1))),
            'cp_best_valid_fitb_acc': float(cp_checkpoint.get('best_valid_fitb_acc', float('nan'))),
            'cp_valid_auc_at_best_fitb': float(cp_checkpoint.get('valid_auc_at_best_fitb', float('nan'))),
            'cp_test_auc': float(cp_test_auc),
            'cp_test_fitb_acc': float(cp_test_fitb),
            'or_best_epoch': int(or_checkpoint.get('epoch', or_checkpoint.get('best_epoch', -1))),
            'or_best_valid_fitb_acc': float(or_checkpoint.get('best_valid_fitb_acc', float('nan'))),
            'or_test_fitb_acc': float(or_test_fitb),
            'cp_checkpoint': str(cp_ckpt_path),
            'or_checkpoint': str(or_ckpt_path),
        }
        seed_summary.update(or_summary)
        all_seed_rows.append(seed_summary)
        all_or_rows.extend(or_rows)

        write_csv(result_dir / 'seed_summary.csv', [seed_summary])
        write_csv(result_dir / 'or_recall_rowlevel.csv', or_rows)

        config = {
            'variant': variant_key,
            'seed': seed,
            'pipeline': 'CP binary compatibility pretraining -> OR retrieval fine-tuning',
            'checkpoint_selection': 'CP and OR checkpoints selected by validation FITB accuracy; train up to 100 epochs without early stopping',
            'cp_checkpoint': str(cp_ckpt_path),
            'or_checkpoint': str(or_ckpt_path),
            'text_feature_file': feature_file,
            'splits': {
                'train': 'PO-D train',
                'valid': 'PO-D valid',
                'test': 'PO-D test',
            },
            'seed_summary': seed_summary,
        }
        (model_dir / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')

seed_summary_path = RESULT_ROOT / 'A30_second_model_two_tower_seed_summary.csv'
rowlevel_path = RESULT_ROOT / 'A31_second_model_two_tower_or_rowlevel.csv'
write_csv(seed_summary_path, all_seed_rows)
write_csv(rowlevel_path, all_or_rows)

print('\n[完成] CP→OR seed-level summary')
for row in all_seed_rows:
    print(json.dumps(row, ensure_ascii=False, indent=2))



[original_text] outfit text embedding 筆數：68,306
  CP train: usable=33,990, missing=0
  CP valid: usable=6,000, missing=0
  CP test: usable=30,290, missing=0
  FITB train: usable=16,995, missing=0
  FITB valid: usable=3,000, missing=0
  FITB test: usable=15,145, missing=0

[CP 訓練] variant=original_text, seed=1, train=33,990, valid=6,000


CP train original_text seed=1 epoch=1:   0%|          | 0/133 [00:00<?, ?it/s]

CP FITB:   0%|          | 0/3000 [00:00<?, ?it/s]

CP epoch=01 loss=0.6526 valid_AUC=0.8040 valid_FITB=0.4137 lr=1.00e-03


CP train original_text seed=1 epoch=2:   0%|          | 0/133 [00:00<?, ?it/s]

...

OR train context_aware_description seed=5 epoch=100:   0%|          | 0/67 [00:00<?, ?it/s]

OR epoch=100 loss=0.0567 valid_FITB=0.6143 lr=1.53e-08


encode candidate pools [context_aware_description seed=5]:   0%|          | 0/19 [00:00<?, ?it/s]

OR/CIR recall [context_aware_description seed=5]:   0%|          | 0/15145 [00:00<?, ?it/s]

[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/context_aware_description/seed_5/seed_summary.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/context_aware_description/seed_5/or_recall_rowlevel.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/A30_second_model_two_tower_seed_summary.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/A31_second_model_two_tower_or_rowlevel.csv

[完成] CP→OR seed-level summary
{
  "variant": "original_text",
  "seed": 1,
  "cp_best_epoch": 13,
  "cp_best_valid_fitb_acc": 0.5806666666666667,
  "cp_valid_auc_at_best_fitb": 0.9130273333333334,
  "cp_test_auc": 0.8780349697613288,
  "cp_test_fitb_

In [13]:
# ------------------------------------------------------------
# 5 seeds mean/std 彙整
# ------------------------------------------------------------
if all_seed_rows:
    metrics = [
        'cp_best_valid_fitb_acc',
        'cp_valid_auc_at_best_fitb',
        'cp_test_auc',
        'cp_test_fitb_acc',
        'or_best_valid_fitb_acc',
        'or_test_fitb_acc',
        'mean_rank',
        'median_rank',
    ] + [f'recall@{k}' for k in K_LIST]

    aggregate_rows = []
    for variant_key in TEXT_VARIANTS.keys():
        rows = [row for row in all_seed_rows if row['variant'] == variant_key]
        out = {
            'variant': variant_key,
            'n_seeds': len(rows),
            'seeds': ','.join(str(row['seed']) for row in rows),
            'test_fitb_questions': rows[0]['test_fitb_questions'] if rows else '',
            'or_recall_evaluable_pairs': rows[0]['or_recall_evaluable_pairs'] if rows else '',
        }
        for metric in metrics:
            values = [float(row[metric]) for row in rows]
            out[f'{metric}_mean'] = float(np.mean(values)) if values else float('nan')
            out[f'{metric}_std'] = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
            out[f'{metric}_mean±std'] = mean_std_text(values)
        aggregate_rows.append(out)

    aggregate_path = RESULT_ROOT / 'A32_second_model_two_tower_mean_std.csv'
    write_csv(aggregate_path, aggregate_rows)

    interpretation_lines = [
        'P15 第二模型補充驗證說明',
        '',
        '本實驗將第二模型改為 CP→OR 兩階段流程，並與主模型設定對齊為最多訓練 100 epochs。',
        '第一階段使用 compatibility_train.txt 訓練 binary compatibility prediction model；每個 epoch 監控 validation AUC，但以 validation FITB Accuracy 更新 cp_best_model.pt。',
        'CP 最佳 checkpoint 於 compatibility_test.txt 回報 AUC，並於 fill_in_blank_test.json 回報 FITB Accuracy。',
        '第二階段將 CP model 的 item encoder 與 outfit text encoder 遷移至 OR retrieval model，再以 validation FITB Accuracy 更新最佳 OR checkpoint，並於 OR/CIR candidate pool 回報 Recall@K。',
        'original_text 與 context_aware_description 兩組條件分開訓練，checkpoint 與結果皆儲存在 second_model_runs/fashionclip_two_tower_fitb_or。',
        '',
        f'CP test pairs：{len(compat_examples_by_split["test"]):,}',
        f'FITB test questions：{len(fitb_examples_by_split["test"]):,}',
        f'OR/CIR Recall evaluable query-target pairs：{aggregate_rows[0]["or_recall_evaluable_pairs"]:,}' if aggregate_rows else '',
        '',
        '論文定位：此模型作為跨架構補充驗證，用來檢查情境感知描述的改善趨勢是否仍存在於另一個 CP→OR 訓練流程中；主結果仍以 Hybrid Attention Layer 為準。',
    ]
    interpretation_path = RESULT_ROOT / 'A33_second_model_two_tower_interpretation.txt'
    interpretation_path.write_text('\n'.join(interpretation_lines), encoding='utf-8')
    print(f'[寫入] {interpretation_path}')

    print('\n[彙整結果]')
    for row in aggregate_rows:
        print(json.dumps(row, ensure_ascii=False, indent=2))
else:
    print('沒有可彙整的 seed 結果。')


[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/A32_second_model_two_tower_mean_std.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/A33_second_model_two_tower_interpretation.txt

[彙整結果]
{
  "variant": "original_text",
  "n_seeds": 5,
  "seeds": "1,2,3,4,5",
  "test_fitb_questions": 15145,
  "or_recall_evaluable_pairs": 9311,
  "cp_best_valid_fitb_acc_mean": 0.578,
  "cp_best_valid_fitb_acc_std": 0.00716085035298028,
  "cp_best_valid_fitb_acc_mean?std": "0.5780 ± 0.0072",
  "cp_valid_auc_at_best_fitb_mean": 0.9134421555555555,
  "cp_valid_auc_at_best_fitb_std": 0.0037039019195477256,
  "cp_valid_auc_at_best_fitb_mean?std": "0.9134 ± 0.0037",
  "cp_test_auc_mean": 0.8796080062858855,
  "cp_test_auc_std": 0.0028953247296642694,
  "cp_test_auc_mean?std": "0.8796 ± 0.0029",
  "cp_test_fitb_acc_mean": 0.54596

In [14]:
# ------------------------------------------------------------
# 輸出位置確認
# ------------------------------------------------------------
print('CP checkpoint：')
print(MODEL_ROOT / '<variant>' / '<seed>' / 'cp_best_model.pt')
print('\nOR checkpoint：')
print(MODEL_ROOT / '<variant>' / '<seed>' / 'best_model.pt')
print('\n主要輸出檔案：')
for path in [
    RESULT_ROOT / 'A30_second_model_two_tower_seed_summary.csv',
    RESULT_ROOT / 'A31_second_model_two_tower_or_rowlevel.csv',
    RESULT_ROOT / 'A32_second_model_two_tower_mean_std.csv',
    RESULT_ROOT / 'A33_second_model_two_tower_interpretation.txt',
]:
    print(path)

CP checkpoint：
/home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/models/<variant>/<seed>/cp_best_model.pt

OR checkpoint：
/home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/models/<variant>/<seed>/best_model.pt

主要輸出檔案：
/home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/A30_second_model_two_tower_seed_summary.csv
/home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/A31_second_model_two_tower_or_rowlevel.csv
/home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fashionclip_two_tower_fitb_or/results/A32_second_model_two_tower_mean_std.csv
/home/ester/valid_description/experiment data/journal_figures_tables_20260629/second_model_runs/fash

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. 讀取既有結果
# ------------------------------------------------------------

ROOT = Path("/home/ester/valid_description")
JOURNAL_DIR = ROOT / "experiment data" / "journal_figures_tables_20260629"

MAIN_PATH = JOURNAL_DIR / "derived_audits" / "A14_statistical_rigor_ci_adjusted_p_effect_sizes.csv"

SECOND_RESULT_DIR = (
    JOURNAL_DIR
    / "second_model_runs"
    / "fashionclip_two_tower_fitb_or"
    / "results"
)

SECOND_MEAN_PATH = SECOND_RESULT_DIR / "A32_second_model_two_tower_mean_std.csv"
SECOND_SEED_PATH = SECOND_RESULT_DIR / "A30_second_model_two_tower_seed_summary.csv"

main_df = pd.read_csv(MAIN_PATH)
second_mean_df = pd.read_csv(SECOND_MEAN_PATH)
second_seed_df = pd.read_csv(SECOND_SEED_PATH)

# ------------------------------------------------------------
# 2. 格式工具
# ------------------------------------------------------------

def mean_std(mean, std, digits=4):
    return f"{float(mean):.{digits}f} ± {float(std):.{digits}f}"

def signed(x, digits=4):
    return f"{float(x):+.{digits}f}"

def ci_text(low, high, digits=4):
    return f"[{float(low):.{digits}f}, {float(high):.{digits}f}]"

def paired_ci_second(metric):
    pivot = second_seed_df.pivot(
        index="seed",
        columns="variant",
        values=metric
    )

    diff = pivot["context_aware_description"] - pivot["original_text"]
    n = len(diff)

    # df=4 時 95% CI 的 t critical 約為 2.776；若目前只跑 seed1，CI 會以 ±0 呈現。
    if n <= 1:
        mean_diff = diff.mean()
        return mean_diff, mean_diff, mean_diff

    t_crit = 2.776
    mean_diff = diff.mean()
    se = diff.std(ddof=1) / np.sqrt(n)

    return mean_diff, mean_diff - t_crit * se, mean_diff + t_crit * se

# ------------------------------------------------------------
# 3. 第二模型欄位自動對應
# ------------------------------------------------------------

old = second_mean_df[second_mean_df["variant"] == "original_text"].iloc[0]
new = second_mean_df[second_mean_df["variant"] == "context_aware_description"].iloc[0]

# 新版 CP→OR 第二模型欄位
SECOND_CP_AUC = "cp_test_auc"
SECOND_CP_FITB = "cp_test_fitb_acc"
SECOND_OR_FITB = "or_test_fitb_acc"

# ------------------------------------------------------------
# 4. 表 1：CP / FITB 任務比較
# ------------------------------------------------------------

cp_rows = []

# 主模型：CP AUC / FITB
for metric_key, metric_name in [
    ("auc", "AUC"),
    ("fitb_acc", "FITB Accuracy"),
]:
    r = main_df[
        (main_df["task"] == "CP") &
        (main_df["metric"] == metric_key)
    ].iloc[0]

    cp_rows.append({
        "Model": "Hybrid Attention",
        "Metric": metric_name,
        "Original text": mean_std(r["baseline_mean"], r["baseline_std"]),
        "Context-aware": mean_std(r["proposed_mean"], r["proposed_std"]),
        "Δ": signed(r["mean_difference"]),
        "95% CI": ci_text(r["ci95_low"], r["ci95_high"]),
    })

# 第二模型：CP AUC / CP FITB
for metric_key, metric_name in [
    (SECOND_CP_AUC, "AUC"),
    (SECOND_CP_FITB, "FITB Accuracy"),
]:
    diff, low, high = paired_ci_second(metric_key)

    cp_rows.append({
        "Model": "Two-Tower",
        "Metric": metric_name,
        "Original text": mean_std(
            old[f"{metric_key}_mean"],
            old[f"{metric_key}_std"],
        ),
        "Context-aware": mean_std(
            new[f"{metric_key}_mean"],
            new[f"{metric_key}_std"],
        ),
        "Δ": signed(diff),
        "95% CI": ci_text(low, high),
    })

cp_table = pd.DataFrame(cp_rows)

print("Table. CP / FITB task comparison")
display(cp_table)

# ------------------------------------------------------------
# 5. 表 2：OR / CIR 任務比較
# ------------------------------------------------------------

or_rows = []

# 主模型：CIR Recall
for metric_key, metric_name in [
    ("recall_at_10", "Recall@10"),
    ("recall_at_30", "Recall@30"),
    ("recall_at_50", "Recall@50"),
]:
    r = main_df[
        (main_df["task"] == "CIR") &
        (main_df["metric"] == metric_key)
    ].iloc[0]

    or_rows.append({
        "Model": "Hybrid Attention",
        "Metric": metric_name,
        "Original text": mean_std(r["baseline_mean"], r["baseline_std"]),
        "Context-aware": mean_std(r["proposed_mean"], r["proposed_std"]),
        "Δ": signed(r["mean_difference"]),
        "95% CI": ci_text(r["ci95_low"], r["ci95_high"]),
    })

# 第二模型：OR Recall
for metric_key, metric_name in [
    ("recall@10", "Recall@10"),
    ("recall@30", "Recall@30"),
    ("recall@50", "Recall@50"),
]:
    diff, low, high = paired_ci_second(metric_key)

    or_rows.append({
        "Model": "Two-Tower",
        "Metric": metric_name,
        "Original text": mean_std(
            old[f"{metric_key}_mean"],
            old[f"{metric_key}_std"],
        ),
        "Context-aware": mean_std(
            new[f"{metric_key}_mean"],
            new[f"{metric_key}_std"],
        ),
        "Δ": signed(diff),
        "95% CI": ci_text(low, high),
    })

or_table = pd.DataFrame(or_rows)

print("Table. OR / CIR task comparison")
display(or_table)

Table. CP / FITB task comparison


,Model,Metric,Original text,Context-aware,Δ,95% CI
0,Hybrid Attention,AUC,0.9292 ± 0.0013,0.9454 ± 0.0013,+0.0162,"[0.0157, 0.0167]"
1,Hybrid Attention,FITB Accuracy,0.6374 ± 0.0037,0.6478 ± 0.0021,+0.0103,"[0.0054, 0.0153]"
2,Two-Tower,AUC,0.8796 ± 0.0029,0.9174 ± 0.0033,+0.0378,"[0.0337, 0.0420]"
3,Two-Tower,FITB Accuracy,0.5460 ± 0.0051,0.5382 ± 0.0050,-0.0077,"[-0.0180, 0.0025]"


Table. OR / CIR task comparison


,Model,Metric,Original text,Context-aware,Δ,95% CI
0,Hybrid Attention,Recall@10,0.0780 ± 0.0010,0.0921 ± 0.0035,+0.0141,"[0.0101, 0.0182]"
1,Hybrid Attention,Recall@30,0.1677 ± 0.0045,0.1874 ± 0.0038,+0.0197,"[0.0153, 0.0241]"
2,Hybrid Attention,Recall@50,0.2303 ± 0.0035,0.2518 ± 0.0038,+0.0215,"[0.0161, 0.0269]"
3,Two-Tower,Recall@10,0.0378 ± 0.0008,0.0479 ± 0.0017,+0.0102,"[0.0087, 0.0117]"
4,Two-Tower,Recall@30,0.0950 ± 0.0017,0.1103 ± 0.0030,+0.0153,"[0.0127, 0.0179]"
5,Two-Tower,Recall@50,0.1419 ± 0.0026,0.1589 ± 0.0046,+0.0170,"[0.0139, 0.0200]"
